# 02 GlocalIB Pre-training
This notebook implements the GlocalIB pre-training objective. It can be run in two modes:
1. `glocal_ib`: Full model with learnable beta.
2. `glocal_beta0`: Ablation study with beta=0 (alignment only).

In [1]:
import sys
import os
import torch
import random
import wandb
from torch.optim import AdamW

sys.path.append("..")
from src.data import load_ecthr, mask_paragraphs
from src.model import GlocalIBModel
from src.loss import glocal_ib_loss

# --- CONFIGURATION ---
CONDITION      = "glocal_ib"   # Options: "glocal_ib", "glocal_beta0"
DISABLE_IB     = (CONDITION == "glocal_beta0")
EPOCHS         = 5
BATCH_SIZE     = 4             # Recommended for 24GB-32GB VRAM. Set to 16 for Spark 128GB.
LR             = 1e-5
CHECKPOINT_DIR = "../checkpoints"
DEVICE         = "cuda" if torch.cuda.is_available() else "cpu"
WANDB_PROJECT  = "glocal-nlp"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Running Condition: {CONDITION} on {DEVICE}")

Running Condition: glocal_ib on cpu


In [2]:
wandb.init(project=WANDB_PROJECT, name=CONDITION, config={
    "condition": CONDITION, 
    "epochs": EPOCHS, 
    "batch_size": BATCH_SIZE, 
    "lr": LR,
    "disable_ib": DISABLE_IB
})

print("Loading data and initializing model...")
dataset = load_ecthr()
train_data = dataset["train"]
model = GlocalIBModel(device=DEVICE)
optimizer = AdamW(model.parameters(), lr=LR)

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose 'Create a W&B account'
wandb: Create an account here: https://wandb.ai/authorize?signup=true&ref=models
wandb: After creating your account, create a new API key and store it securely.
wandb: Paste your API key and hit enter:wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /Users/brandomattivi/.netrc
wandb: Currently logged in as: brandomattivi (bmattivi03) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Loading data and initializing model...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [3]:
print("Starting Pre-training...")
for epoch in range(EPOCHS):
    model.train()
    indices = list(range(len(train_data)))
    random.shuffle(indices)
    epoch_loss = 0.0
    
    # Batch loop
    for i in range(0, len(indices), BATCH_SIZE):
        batch_idx = indices[i : i + BATCH_SIZE]
        full_batch = [train_data[j]["text"] for j in batch_idx]
        masked_batch = [mask_paragraphs(train_data[j]["text"]) for j in batch_idx]

        optimizer.zero_grad()
        
        # Forward
        Z_prime, mu, sigma, Z_proj, beta = model(full_batch, masked_batch)
        
        # Loss
        loss, l_align, l_compress = glocal_ib_loss(
            Z_proj, Z_prime, mu, sigma, beta, disable_ib=DISABLE_IB
        )
        
        # Backward
        loss.backward()
        optimizer.step()

        # Log
        if (i // BATCH_SIZE) % 10 == 0:
            wandb.log({
                "loss": loss.item(), 
                "l_align": l_align.item(),
                "l_compress": l_compress.item(), 
                "beta": beta.item(), 
                "epoch": epoch,
                "step": i // BATCH_SIZE
            })
            
        epoch_loss += loss.item()

    avg_loss = epoch_loss / (len(indices) / BATCH_SIZE)
    print(f"Epoch {epoch+1}/{EPOCHS} | Avg Loss: {avg_loss:.4f} | Beta: {beta.item():.4f}")

    # Save Checkpoint
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'beta': beta.item(),
    }, f"{CHECKPOINT_DIR}/{CONDITION}_epoch{epoch+1}.pt")

wandb.finish()
print("Training Complete.")

Starting Pre-training...


KeyboardInterrupt: 

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x309227b90>> (for post_run_cell), with arguments args (<ExecutionResult object at 119fb8e10, execution_count=3 error_before_exec=None error_in_exec= info=<ExecutionInfo object at 119fb89d0, raw_cell="print("Starting Pre-training...")
for epoch in ran.." transformed_cell="print("Starting Pre-training...")
for epoch in ran.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell:/Users/brandomattivi/Documents/UNI/UNIBZ/MSC/2nd%20semester/nlp/GlocalDoc/notebooks/02_pretrain_glocal.ipynb#W3sZmlsZQ%3D%3D> result=None>,),kwargs {}:


ConnectionResetError: Connection lost